# Chat Models

The `chat_models.py` module defines the shared interface and execution machinery for conversational language models.

It provides helpers for combining streamed generation chunks, the abstract `BaseChatModel` interface, synchronous and asynchronous invocation, token streaming, versioned event streaming, caching, tracing, rate limiting, model profiles, tool binding, structured output, and the backward-compatible `SimpleChatModel` implementation.

### Functions

1. `generate_from_stream`: Combines a synchronous stream of chat-generation chunks into one `ChatResult`.

   The first chunk is accumulated with all remaining chunks. Its final message chunk is converted into a complete message, while generation information is preserved.

   A `ValueError` is raised when the stream does not contain any generation chunks.

   * **Syntax:**
     ```python
     generate_from_stream(
         stream: Iterator[
             ChatGenerationChunk
         ] # Streamed chat-generation chunks
     ) -> ChatResult
     ```

2. `agenerate_from_stream`: Asynchronously combines a stream of chat-generation chunks into one `ChatResult`.

   The asynchronous stream is collected into a list, after which `generate_from_stream` is executed through an executor. The same empty-stream `ValueError` may therefore be raised.

   * **Syntax:**
     ```python
     async agenerate_from_stream(
         stream: AsyncIterator[
             ChatGenerationChunk
         ] # Asynchronously streamed chat-generation chunks
     ) -> ChatResult
     ```

# BaseChatModel

`BaseChatModel` is the abstract base class for LangChain chat-model integrations.

It accepts strings, message-like sequences, and `PromptValue` objects. It returns `AIMessage` values for normal invocation, `AIMessageChunk` values for token streaming, `LLMResult` values for batched generation, and version-specific event-stream objects for tracing-oriented streaming.

Concrete integrations must implement `_generate` and `_llm_type`. Native streaming, asynchronous generation, tool binding, structured output, LangSmith parameters, and model-profile resolution may be implemented or overridden as supported by the provider.

## Bases

- `BaseLanguageModel[AIMessage]`
- `ABC`

## Attributes

1. `rate_limiter`: Stores an optional rate limiter applied before provider requests.

   Synchronous streaming acquires it in blocking mode. Asynchronous streaming uses its asynchronous acquisition interface.

   * **Type:**
     ```python
     rate_limiter: BaseRateLimiter | None = Field(
         default=None,
         exclude=True
     )
     ```

2. `disable_streaming`: Controls whether streaming should be bypassed.

   - `False` allows streaming when the model supports it.
   - `True` always routes streaming calls through non-streaming invocation.
   - `"tool_calling"` bypasses streaming only when tools are supplied.

   Explicitly false `stream` invocation parameters and provider-level `streaming=False` settings may also disable streaming for an individual call.

   * **Type:**
     ```python
     disable_streaming: bool
     | Literal[
         "tool_calling"
     ] = False
     ```

3. `output_version`: Controls the format stored in `AIMessage.content`.

   `"v0"` keeps provider-specific content, while `"v1"` stores standardized content blocks. The default may be read from the `LC_OUTPUT_VERSION` environment variable.

   * **Type:**
     ```python
     output_version: str | None = Field(
         default_factory=from_env(
             "LC_OUTPUT_VERSION",
             default=None
         )
     )
     ```

4. `profile`: Stores optional model-capability information.

   Provider integrations may populate this automatically during model validation. Profiles may describe supported modalities, context-window limits, tool calling, structured output, and other capabilities.

   This field is a beta feature.

   * **Type:**
     ```python
     profile: ModelProfile | None = Field(
         default=None,
         exclude=True
     )
     ```

## Configuration

1. `model_config`: Allows arbitrary Python types in the Pydantic model.
   * **Definition:**
     ```python
     model_config = ConfigDict(
         arbitrary_types_allowed=True
     )
     ```

### Properties

1. `OutputType`: Returns the concrete Runnable output type for the chat model.
   * **Type:**
     ```python
     OutputType: Any
     ```

   * **Value:**
     ```python
     AnyMessage
     ```

2. `_llm_type`: Returns the provider-specific model type used for identification and serialization.

   Concrete subclasses must implement this abstract property.

   * **Type:**
     ```python
     _llm_type: str
     ```

### Methods

1. `_resolve_model_profile`: Returns a default capability profile for the concrete model.

   Provider integrations should override this protected method rather than replacing the model-profile validator. The base implementation returns `None`.

   Errors raised while resolving an optional profile are suppressed during model construction.

   * **Syntax:**
     ```python
     _resolve_model_profile(
         self
     ) -> ModelProfile | None
     ```

2. `invoke`: Performs one synchronous chat-model invocation.

   The input is converted to a `PromptValue`, generation is delegated to `generate_prompt`, and the first generated message is returned.

   Runnable configuration supplies callbacks, tags, metadata, run name, and an optional run ID.

   * **Syntax:**
     ```python
     invoke(
         self,
         input: LanguageModelInput, # String, prompt value, or message-like sequence
         config: RunnableConfig | None = None, # Runnable configuration
         *,
         stop: list[str] | None = None, # Stop substrings
         **kwargs: Any # Provider-specific invocation parameters
     ) -> AIMessage
     ```

3. `ainvoke`: Performs one asynchronous chat-model invocation.

   It delegates to `agenerate_prompt` and returns the first generated message. Configuration handling matches `invoke`.

   * **Syntax:**
     ```python
     async ainvoke(
         self,
         input: LanguageModelInput, # String, prompt value, or message-like sequence
         config: RunnableConfig | None = None, # Runnable configuration
         *,
         stop: list[str] | None = None, # Stop substrings
         **kwargs: Any # Provider-specific invocation parameters
     ) -> AIMessage
     ```

4. `stream`: Streams `AIMessageChunk` values synchronously.

   When native streaming is unavailable or disabled, the method yields the complete result of `invoke` as a single chunk-compatible value.

   During native streaming, it:

   - Creates a chat-model trace.
   - Acquires the optional rate limiter.
   - Calls `_stream`.
   - Assigns a stable message ID when absent.
   - Adds generation and response metadata.
   - Applies the configured output version.
   - Sends token callbacks.
   - Accumulates chunks for the final completion callback.

   Provider exceptions trigger the LLM-error callback with partial-generation information before being re-raised. A `ValueError` is raised when native streaming produces no chunks.

   * **Syntax:**
     ```python
     stream(
         self,
         input: LanguageModelInput, # String, prompt value, or message-like sequence
         config: RunnableConfig | None = None, # Runnable configuration
         *,
         stop: list[str] | None = None, # Stop substrings
         **kwargs: Any # Provider-specific streaming parameters
     ) -> Iterator[
         AIMessageChunk
     ]
     ```

5. `astream`: Streams `AIMessageChunk` values asynchronously.

   When asynchronous and synchronous streaming are both unavailable or streaming is disabled, it yields the complete `ainvoke` result as one value.

   Native asynchronous streaming follows the same tracing, rate-limiting, metadata, output-version, callback, error, and finalization behaviour as `stream`.

   The default `_astream` implementation can bridge a synchronous `_stream` iterator through an executor.

   * **Syntax:**
     ```python
     async astream(
         self,
         input: LanguageModelInput, # String, prompt value, or message-like sequence
         config: RunnableConfig | None = None, # Runnable configuration
         *,
         stop: list[str] | None = None, # Stop substrings
         **kwargs: Any # Provider-specific streaming parameters
     ) -> AsyncIterator[
         AIMessageChunk
     ]
     ```

6. `stream_events`: Streams versioned chat-model events synchronously.

   For versions `"v1"` and `"v2"`, the method delegates to the standard Runnable event-stream implementation and returns an iterator of `StreamEvent` dictionaries.

   Version `"v3"` returns a `ChatModelStream` with replayable raw events and typed text, reasoning, tool-call, and output projections. Version 3 is beta and always produces version 1 standardized content blocks in its final message.

   The `stop` parameter is used directly by version 3 and is forwarded through the Runnable implementation for earlier versions.

   * **Syntax:**
     ```python
     stream_events(
         self,
         input: LanguageModelInput, # Chat-model input
         config: RunnableConfig | None = None, # Runnable configuration
         *,
         version: Literal[
             "v1",
             "v2",
             "v3"
         ] = "v2", # Event-stream protocol version
         stop: list[str] | None = None, # Stop substrings
         **kwargs: Any # Additional event-stream or provider parameters
     ) -> Iterator[
         StreamEvent
     ] | ChatModelStream
     ```

7. `astream_events`: Returns asynchronous versioned chat-model events.

   For versions `"v1"` and `"v2"`, it returns the asynchronous iterator provided by the standard Runnable event-stream implementation.

   For version `"v3"`, it returns an awaitable that resolves to `AsyncChatModelStream`. The resulting stream is asynchronously iterable over raw events and exposes awaitable text, reasoning, tool-call, and output projections.

   * **Syntax:**
     ```python
     astream_events(
         self,
         input: LanguageModelInput, # Chat-model input
         config: RunnableConfig | None = None, # Runnable configuration
         *,
         version: Literal[
             "v1",
             "v2",
             "v3"
         ] = "v2", # Event-stream protocol version
         stop: list[str] | None = None, # Stop substrings
         **kwargs: Any # Additional event-stream or provider parameters
     ) -> AsyncIterator[
         StreamEvent
     ] | Awaitable[
         AsyncChatModelStream
     ]
     ```

8. `_combine_llm_outputs`: Combines provider-level output dictionaries from multiple prompt results.

   The base implementation returns an empty dictionary. Providers may override it when batch results contain mergeable usage or response metadata.

   * **Syntax:**
     ```python
     _combine_llm_outputs(
         self,
         _llm_outputs: list[
             dict[
                 str,
                 Any
             ] | None
         ], / # Provider outputs to combine
     ) -> dict[
         str,
         Any
     ]
     ```

9. `_get_invocation_params`: Returns serialized model parameters combined with stop sequences and invocation-specific keyword arguments.

   The resulting dictionary is used for tracing and cache-key construction.

   * **Syntax:**
     ```python
     _get_invocation_params(
         self,
         stop: list[str] | None = None, # Stop substrings
         **kwargs: Any # Invocation-specific parameters
     ) -> dict[
         str,
         Any
     ]
     ```

10. `_get_ls_params`: Returns standardized LangSmith tracing parameters.

    The base implementation derives a provider name from the concrete class name, sets the model type to `"chat"`, and detects model name, temperature, maximum-token, and stop values from invocation arguments or model attributes.

    Provider integrations should override this protected method to expose stable provider and model identifiers, especially when model attributes use provider-specific names.

    * **Syntax:**
      ```python
      _get_ls_params(
          self,
          stop: list[str] | None = None, # Stop substrings
          **kwargs: Any # Invocation-specific model parameters
      ) -> LangSmithParams
      ```

11. `_get_ls_params_with_defaults`: Returns LangSmith parameters and ensures the integration is identified as a LangChain chat model.
    * **Syntax:**
      ```python
      _get_ls_params_with_defaults(
          self,
          stop: list[str] | None = None, # Stop substrings
          **kwargs: Any # Invocation-specific model parameters
      ) -> LangSmithParams
      ```

12. `_get_llm_string`: Returns the model-and-invocation string used as a cache key.

    Serializable models use a cleaned serialized representation plus sorted invocation parameters. Non-serializable models use sorted identifying and invocation parameters.

    * **Syntax:**
      ```python
      _get_llm_string(
          self,
          stop: list[str] | None = None, # Stop substrings
          **kwargs: Any # Invocation-specific parameters
      ) -> str
      ```

13. `generate`: Generates results synchronously for multiple message batches.

    One chat-model run is opened for each input message list. Inputs are normalized, generated through the cache-aware execution path, and combined into one `LLMResult`.

    Callback configuration combines invocation-level and model-level callbacks, tags, and metadata. LangSmith parameters and filtered invocation parameters are attached as inheritable trace metadata.

    On failure, the corresponding run receives an LLM-error callback before the exception is re-raised. Successful runs receive LLM-end callbacks and their run IDs are attached to the result.

    * **Syntax:**
      ```python
      generate(
          self,
          messages: list[
              list[
                  BaseMessage
              ]
          ], # Message batches
          stop: list[str] | None = None, # Stop substrings
          callbacks: Callbacks = None, # Invocation callbacks
          *,
          tags: list[str] | None = None, # Trace tags
          metadata: dict[
              str,
              Any
          ] | None = None, # Trace metadata
          run_name: str | None = None, # Optional run name
          run_id: UUID | None = None, # Optional run identifier
          **kwargs: Any # Provider-specific generation parameters
      ) -> LLMResult
      ```

14. `agenerate`: Generates results asynchronously for multiple message batches.

    Individual prompt operations are scheduled concurrently through `asyncio.gather`. Successful and failed runs receive their appropriate callbacks.

    When one or more operations fail, successful runs are finalized before the first collected exception is raised.

    * **Syntax:**
      ```python
      async agenerate(
          self,
          messages: list[
              list[
                  BaseMessage
              ]
          ], # Message batches
          stop: list[str] | None = None, # Stop substrings
          callbacks: Callbacks = None, # Invocation callbacks
          *,
          tags: list[str] | None = None, # Trace tags
          metadata: dict[
              str,
              Any
          ] | None = None, # Trace metadata
          run_name: str | None = None, # Optional run name
          run_id: UUID | None = None, # Optional run identifier
          **kwargs: Any # Provider-specific generation parameters
      ) -> LLMResult
      ```

15. `generate_prompt`: Converts each prompt value to messages and delegates to `generate`.
    * **Syntax:**
      ```python
      generate_prompt(
          self,
          prompts: list[
              PromptValue
          ], # Prompt values to generate
          stop: list[str] | None = None, # Stop substrings
          callbacks: Callbacks = None, # Invocation callbacks
          **kwargs: Any # Provider-specific generation parameters
      ) -> LLMResult
      ```

16. `agenerate_prompt`: Converts each prompt value to messages and delegates to `agenerate`.
    * **Syntax:**
      ```python
      async agenerate_prompt(
          self,
          prompts: list[
              PromptValue
          ], # Prompt values to generate
          stop: list[str] | None = None, # Stop substrings
          callbacks: Callbacks = None, # Invocation callbacks
          **kwargs: Any # Provider-specific generation parameters
      ) -> LLMResult
      ```

17. `_generate`: Generates one complete `ChatResult` synchronously.

    This is the primary required implementation hook for concrete chat models.

    * **Syntax:**
      ```python
      @abstractmethod
      _generate(
          self,
          messages: list[
              BaseMessage
          ], # Input messages
          stop: list[str] | None = None, # Stop substrings
          run_manager: CallbackManagerForLLMRun | None = None, # Synchronous run manager
          **kwargs: Any # Provider-specific generation parameters
      ) -> ChatResult
      ```

18. `_agenerate`: Generates one complete `ChatResult` asynchronously.

    The base implementation runs `_generate` in an executor and converts an asynchronous run manager to its synchronous view. Providers may override this method with a native asynchronous API.

    * **Syntax:**
      ```python
      async _agenerate(
          self,
          messages: list[
              BaseMessage
          ], # Input messages
          stop: list[str] | None = None, # Stop substrings
          run_manager: AsyncCallbackManagerForLLMRun | None = None, # Async run manager
          **kwargs: Any # Provider-specific generation parameters
      ) -> ChatResult
      ```

19. `_stream`: Streams provider `ChatGenerationChunk` values synchronously.

    Providers may override this optional protected hook. The base implementation raises `NotImplementedError`.

    * **Syntax:**
      ```python
      _stream(
          self,
          messages: list[
              BaseMessage
          ], # Input messages
          stop: list[str] | None = None, # Stop substrings
          run_manager: CallbackManagerForLLMRun | None = None, # Synchronous run manager
          **kwargs: Any # Provider-specific streaming parameters
      ) -> Iterator[
          ChatGenerationChunk
      ]
      ```

20. `_astream`: Streams provider `ChatGenerationChunk` values asynchronously.

    The base implementation obtains the synchronous `_stream` iterator in an executor and repeatedly advances it through the executor. Providers may override it with native asynchronous streaming.

    * **Syntax:**
      ```python
      async _astream(
          self,
          messages: list[
              BaseMessage
          ], # Input messages
          stop: list[str] | None = None, # Stop substrings
          run_manager: AsyncCallbackManagerForLLMRun | None = None, # Async run manager
          **kwargs: Any # Provider-specific streaming parameters
      ) -> AsyncIterator[
          ChatGenerationChunk
      ]
      ```

21. `dict`: Returns the identifying dictionary for the model.

    This method is deprecated in favour of `asdict` and is scheduled for removal in version `2.0.0`.

    * **Syntax:**
      ```python
      dict(
          self,
          **_kwargs: Any # Deprecated compatibility arguments
      ) -> dict[
          str,
          Any
      ]
      ```

22. `asdict`: Returns a dictionary containing the identifying parameters and the `_llm_type` value.
    * **Syntax:**
      ```python
      asdict(
          self
      ) -> dict[
          str,
          Any
      ]
      ```

23. `bind`: Returns a chat-model-specific Runnable binding with fixed keyword arguments.

    Unlike the generic Runnable binding, the returned internal binding preserves the typed overloads for version 3 `stream_events` and `astream_events`.

    * **Syntax:**
      ```python
      bind(
          self,
          **kwargs: Any # Keyword arguments fixed on the returned Runnable
      ) -> _ChatModelBinding
      ```

24. `bind_tools`: Returns a Runnable configured to expose tools to the model.

    The accepted tools may be dictionaries, Python types, callables, or `BaseTool` objects. `tool_choice` may request a specific tool or permit any bound tool.

    Concrete providers must override this method to support tool calling. The base implementation raises `NotImplementedError`.

    * **Syntax:**
      ```python
      bind_tools(
          self,
          tools: Sequence[
              dict[
                  str,
                  Any
              ]
              | type
              | Callable[
                  ...,
                  Any
              ]
              | BaseTool
          ], # Tools exposed to the model
          *,
          tool_choice: str | None = None, # Tool-selection instruction
          **kwargs: Any # Provider-specific tool-binding parameters
      ) -> Runnable[
          LanguageModelInput,
          AIMessage
      ]
      ```

25. `with_structured_output`: Returns a Runnable that parses model output according to a schema.

    The schema may be an OpenAI tool schema, JSON Schema, `TypedDict` class, or Pydantic model class.

    When the schema is a Pydantic class, parsed output is validated and returned as a model instance. Other schema forms produce dictionaries.

    With `include_raw=False`, parsing errors are raised. With `include_raw=True`, the result contains:

    - `"raw"`: the original model message.
    - `"parsed"`: parsed output or `None`.
    - `"parsing_error"`: the caught parsing exception or `None`.

    The implementation relies on `bind_tools`. A `NotImplementedError` is raised when tool binding is not implemented, and a `ValueError` is raised for unsupported keyword arguments.

    * **Syntax:**
      ```python
      with_structured_output(
          self,
          schema: dict[
              str,
              Any
          ] | type, # Required output schema
          *,
          include_raw: bool = False, # Include raw message and parsing error
          **kwargs: Any # Compatibility options or unsupported arguments
      ) -> Runnable[
          LanguageModelInput,
          dict[
              str,
              Any
          ] | BaseModel
      ]
      ```

## Cache and Streaming Behaviour

The cache-aware generation path supports both normal generation and streaming-triggered generation:

- A configured model cache is used when `cache=True`, when an explicit cache instance is supplied, or when the global cache is enabled.
- Cached legacy `Generation` objects are converted to `ChatGeneration` objects.
- Cache-hit usage metadata has `total_cost` reset to zero when possible.
- Cached messages may be replayed through version 2 protocol handlers so warm-cache and cold-cache event streams remain consistent.
- New generations are written to the cache after successful completion.
- Version 1 callback streaming and version 2 protocol streaming use separate opt-in checks.
- Output chunks are normalized before being returned or traced.

## Tracing Behaviour

Chat-model runs include:

- Serialized model information.
- Formatted input messages.
- Invocation parameters.
- Stop and structured-output options.
- Model and invocation callbacks.
- Tags and metadata.
- LangSmith provider, model, temperature, token, stop, and integration parameters when available.
- Stable run and message identifiers.
- Success or error callbacks containing generated or partial output.

# SimpleChatModel

`SimpleChatModel` is a backward-compatible base class for chat models that return plain text from a simpler `_call` method.

New integrations should generally subclass `BaseChatModel` directly.

## Bases

- `BaseChatModel`

### Methods

1. `_generate`: Calls `_call`, wraps the returned string in an `AIMessage`, and returns one `ChatGeneration` inside a `ChatResult`.
   * **Syntax:**
     ```python
     _generate(
         self,
         messages: list[
             BaseMessage
         ], # Input messages
         stop: list[str] | None = None, # Stop substrings
         run_manager: CallbackManagerForLLMRun | None = None, # Synchronous run manager
         **kwargs: Any # Provider-specific generation parameters
     ) -> ChatResult
     ```

2. `_call`: Returns the model response as a plain string.

   Concrete `SimpleChatModel` subclasses must implement this abstract method.

   * **Syntax:**
     ```python
     @abstractmethod
     _call(
         self,
         messages: list[
             BaseMessage
         ], # Input messages
         stop: list[str] | None = None, # Stop substrings
         run_manager: CallbackManagerForLLMRun | None = None, # Synchronous run manager
         **kwargs: Any # Provider-specific generation parameters
     ) -> str
     ```

3. `_agenerate`: Asynchronously generates a result by running `_generate` in an executor.

   An asynchronous run manager is converted to its synchronous view before delegation.

   * **Syntax:**
     ```python
     async _agenerate(
         self,
         messages: list[
             BaseMessage
         ], # Input messages
         stop: list[str] | None = None, # Stop substrings
         run_manager: AsyncCallbackManagerForLLMRun | None = None, # Async run manager
         **kwargs: Any # Provider-specific generation parameters
     ) -> ChatResult
     ```

## Internal Components Omitted

The following internal implementation details are not documented as standalone public API entries:

- Error-response metadata extraction
- Message formatting and normalization helpers
- Structured-output trace formatting
- Streaming-selection predicates
- Version 2 protocol bridges
- Version 3 source-pump implementations
- Cache-hit protocol replay helpers
- `_ChatModelBinding`
- Representation-cleanup helpers
- Generation metadata merge helpers